# 02. Preprocessing
Cleaning, Normalization, Encoding, and Scaling for ALL data found in /input.

In [ ]:
# Preprocessing Base Configuration
import os
import re
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.rcParams.update({'font.size': 18})
plt.rcParams['axes.titlesize'] = 20
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16
plt.rcParams['legend.fontsize'] = 16
import torch

# --- GLOBAL PATHS ---
BASE_PATH = Path('./input')
PROCESSED_PATH = Path('./processed')
OUTPUT_PATH = Path('./outputs')

PROCESSED_PATH.mkdir(exist_ok=True)
OUTPUT_PATH.mkdir(exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f'Stage: {stage_name}')
print(f'Input Path: {BASE_PATH}')


In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

def sanitize_name(s):
    return re.sub(r'\s+', '_', str(s).strip())

def process_all_data():
    all_processed = {}
    all_labels = set()
    
    # 1. Find all instruments and features
    instruments = [d.name for d in BASE_PATH.iterdir() if d.is_dir() and not d.name.startswith('.')]
    print(f'Found instruments: {instruments}')
    
    # 2. Collect all possible labels first
    for inst in instruments:
        inst_path = BASE_PATH / inst
        features = [d.name for d in inst_path.iterdir() if d.is_dir()]
        for feat in features:
            path_pkl = inst_path / feat / f'extracted_features_{feat}.pkl'
            path_csv = inst_path / feat / f'extracted_features_{feat}.csv'
            
            if path_pkl.exists():
                with open(path_pkl, 'rb') as f: df_temp = pickle.load(f)
            elif path_csv.exists():
                df_temp = pd.read_csv(path_csv)
            else: continue
            
            all_labels.update(df_temp['roman_numeral'].unique())
    
    le = LabelEncoder()
    le.fit(list(all_labels))
    print(f'Total classes: {len(le.classes_)}')
    
    # 3. Process each combination
    for inst in instruments:
        inst_path = BASE_PATH / inst
        features = [d.name for d in inst_path.iterdir() if d.is_dir()]
        for feat in features:
            path_pkl = inst_path / feat / f'extracted_features_{feat}.pkl'
            path_csv = inst_path / feat / f'extracted_features_{feat}.csv'
            
            if path_pkl.exists():
                with open(path_pkl, 'rb') as f: df = pickle.load(f)
            elif path_csv.exists():
                df = pd.read_csv(path_csv)
            else: continue
            
            # Track ID
            if 'track_id' not in df.columns:
                df['track_id'] = df['artist'].map(sanitize_name) + '-' + df['title'].map(sanitize_name)
            
            # Features
            feat_cols = [c for c in df.columns if c not in ['artist', 'title', 'start_time', 'end_time', 'roman_numeral', 'chord', 'label', 'track_id']]
            
            # Scaling
            scaler = StandardScaler()
            df[feat_cols] = scaler.fit_transform(df[feat_cols])
            
            # Encoding
            df['label_idx'] = le.transform(df['roman_numeral'])
            
            # Group into sequences
            sequences = []
            for tid, group in df.groupby('track_id'):
                group = group.sort_values('start_time')
                X = group[feat_cols].values.astype(np.float32)
                y = group['label_idx'].values.astype(np.int64)
                sequences.append({'track_id': tid, 'X': X, 'y': y})
            
            key = f'{inst}_{feat}'
            all_processed[key] = {
                'sequences': sequences,
                'feat_cols': feat_cols,
                'classes': le.classes_.tolist()
            }
            print(f'Processed {key}: {len(sequences)} sequences')
    
    # Save Master File
    out_file = PROCESSED_PATH / 'master_processed_data.pkl'
    with open(out_file, 'wb') as f:
        pickle.dump(all_processed, f)
    print(f'\nSaved MASTER processed data to {out_file}')

process_all_data()